# ISU-GeoBot Thesis — Machine Learning Availability Benchmark
**Authors:** Michael Allan Almario, Christian Paul Simbulan  
**Degree:** BSCS – Data Mining Track, College of Computing Studies, ICT, ISU – Echague

## Overview & Mathematical Formulation (Thesis §3.5.2 & §3.7)
This notebook evaluates the **Random Forest Classifier** against the **Rule-Based Schedule Lookup Baseline** across 97,632 samples from the CCSICT departmental dataset.

### Key Equations Satisfied:
1. **Gini Impurity (Split Criterion):**
$$I_G(p) = 1 - \sum_{i=1}^J p_i^2$$
2. **Overall Accuracy:**
$$\text{Accuracy} = \frac{\sum \text{TP} + \sum \text{TN}}{\text{Total Samples}}$$
3. **Per-Category Precision, Recall & F1:**
$$\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}, \quad \text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}, \quad \text{F1}_c = 2 \times \frac{\text{Precision}_c \times \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$
4. **Macro-Averaged F1-Score:**
$$\text{Macro F1} = \frac{1}{|C|} \sum_{c \in C} \text{F1}_c$$


In [ ]:
import sys
from pathlib import Path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT / 'machine-learning'))
sys.path.insert(0, str(ROOT / 'notebooks'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

from dataset_loader import build_samples
from feature_engineering import CLASS_ORDER, FacultyEncoder, build_vector, feature_names
from metrics_calculator import compute_multiclass_metrics, report_to_markdown_table


## 1. Load Dataset and Pretrained Random Forest Model


In [ ]:
sem_start = date(2026, 8, 10)
sem_end = date(2026, 12, 18)
semester = '2026-2027-1'

print('Building samples from dataset...')
samples = build_samples(semester, sem_start, sem_end, label_source='attendance_derived')
print(f'Total samples: {len(samples):,}')

# Time-based 80/20 train-test split
n = len(samples)
order = np.argsort([s.when for s in samples])
cut = int(n * 0.80)
train_idx, test_idx = order[:cut], order[cut:]
test_samples = [samples[i] for i in test_idx]

# Load trained RF bundle
model_bundle = joblib.load(ROOT / 'machine-learning' / 'saved-models' / 'rf_current.joblib')
rf_model = model_bundle['model']
encoder = model_bundle.get('encoder') or FacultyEncoder().fit(s.pseudonym_id for s in samples)


## 2. Compute Predictions (Random Forest vs. Rule Baseline)


In [ ]:
X_test = np.array([build_vector(s.context, encoder, include_attendance=True) for s in test_samples])
y_test = np.array([s.label for s in test_samples])

# RF Predictions
y_pred_rf = rf_model.predict(X_test)

# Rule-Based Baseline Predictions
y_pred_base = np.array([
    'in_scheduled_class' if s.context.is_scheduled_class else 'unavailable_off_schedule'
    for s in test_samples
])

rf_report = compute_multiclass_metrics(y_test, y_pred_rf, classes=CLASS_ORDER)
base_report = compute_multiclass_metrics(y_test, y_pred_base, classes=CLASS_ORDER)


## 3. Results & Comparative Performance Summary


In [ ]:
comp_df = pd.DataFrame([
    {
        'Model / Architecture': 'Rule-Based Baseline (§3.7)',
        'Accuracy': f'{base_report.accuracy * 100:.2f}%',
        'Macro F1': f'{base_report.macro_f1:.4f}',
        'Consultation F1': f"{base_report.per_class_metrics['available_consultation']['f1_score']:.4f}",
        'Lecture F1': f"{base_report.per_class_metrics['in_scheduled_class']['f1_score']:.4f}",
        'Off-Schedule F1': f"{base_report.per_class_metrics['unavailable_off_schedule']['f1_score']:.4f}"
    },
    {
        'Model / Architecture': 'Enhanced RF Classifier (§3.5.2)',
        'Accuracy': f'{rf_report.accuracy * 100:.2f}%',
        'Macro F1': f'{rf_report.macro_f1:.4f}',
        'Consultation F1': f"{rf_report.per_class_metrics['available_consultation']['f1_score']:.4f}",
        'Lecture F1': f"{rf_report.per_class_metrics['in_scheduled_class']['f1_score']:.4f}",
        'Off-Schedule F1': f"{rf_report.per_class_metrics['unavailable_off_schedule']['f1_score']:.4f}"
    },
    {
        'Model / Architecture': 'Delta (Improvement)',
        'Accuracy': f'+{(rf_report.accuracy - base_report.accuracy) * 100:.2f}%',
        'Macro F1': f'+{(rf_report.macro_f1 - base_report.macro_f1):.4f}',
        'Consultation F1': f"+{rf_report.per_class_metrics['available_consultation']['f1_score']:.4f}",
        'Lecture F1': f"+{(rf_report.per_class_metrics['in_scheduled_class']['f1_score'] - base_report.per_class_metrics['in_scheduled_class']['f1_score']):.4f}",
        'Off-Schedule F1': f"+{(rf_report.per_class_metrics['unavailable_off_schedule']['f1_score'] - base_report.per_class_metrics['unavailable_off_schedule']['f1_score']):.4f}"
    }
])
comp_df


## 4. Visualizations (Confusion Matrix & Feature Importances)


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'notebooks' / 'figures' / 'fig1_confusion_matrix_rf.png')))
display(Image(filename=str(ROOT / 'notebooks' / 'figures' / 'fig3_baseline_vs_rf_comparison.png')))
display(Image(filename=str(ROOT / 'notebooks' / 'figures' / 'fig4_feature_importance.png')))
